# Density-fitted active-space Hamiltonians

Extract CAS(2,3) for LiH using the configured Weigend fitting object. PySCF's
CASCI solver is an independent energy oracle. The QARP adapter transforms only
active MOs and includes the nuclear/frozen-core constant once; it does not run SCF
or CI. Dense output still scales as the fourth power of the active-orbital count.

In [1]:
import numpy as np
from pyscf import ao2mo, gto, lib, mcscf, scf
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.pyscf import active_space_from_mf

lib.num_threads(1)
mf = scf.RHF(gto.M(atom="Li 0 0 0; H 0 0 1.6", basis="cc-pvdz", verbose=0))
mf = mf.density_fit(auxbasis="weigend").run(conv_tol=1e-12)
assert mf.converged
# Select an active/external permutation once, before constructing any operators.
order = list(range(mf.mo_coeff.shape[1]))
order[3], order[4] = order[4], order[3]
mf.mo_coeff = mf.mo_coeff[:, order]
coefficients = mf.mo_coeff.copy()
integrals, onv = active_space_from_mf(mf, active_electrons=2, active_orbitals=3)
assert np.array_equal(mf.mo_coeff, coefficients)
constant, h1, h2 = integrals
assert h2.shape == (3, 3, 3, 3)
hamiltonian = restricted_integrals_to_fermion_operator(*integrals)
matrix = hamiltonian.sparse_matrix(6).toarray()
# Exactly one alpha and one beta electron, with LSB abab indexing.
labels = [
    i for i in range(64) if (i & 0b010101).bit_count() == 1 and (i & 0b101010).bit_count() == 1
]
energy = np.linalg.eigvalsh(matrix[np.ix_(labels, labels)])[0]
print("Active ONV:", onv)
print("Full-MO tensor entries:", mf.mo_coeff.shape[1] ** 4, "active entries:", h2.size)

Active ONV: [1, 1, 0, 0, 0, 0]
Full-MO tensor entries: 130321 active entries: 81


In [2]:
cas = mcscf.CASCI(mf, 3, 2)
cas.canonicalization = False
cas.fcisolver.conv_tol = 1e-12
cas.kernel()
np.testing.assert_allclose(energy, cas.e_tot, rtol=0, atol=1e-9)
print("QARP - DF-CASCI (hartree):", energy - cas.e_tot)

# Use these getters directly when the reference has its own orbitals/settings.
reference_h1, reference_constant = cas.get_h1eff(cas.mo_coeff)
reference_h2 = ao2mo.restore(1, cas.get_h2eff(cas.mo_coeff), cas.ncas)
for actual, expected in zip(
    integrals, (reference_constant, reference_h1, reference_h2), strict=True
):
    np.testing.assert_allclose(actual, expected, rtol=0, atol=1e-10)
assert cas.with_df is mf.with_df

QARP - DF-CASCI (hartree): -8.881784197001252e-16


The full-space restricted/UHF adapters also honor `mf.with_df`, then supplied
`mf._eri`, then conventional molecular integrals. In contrast, generic
`active_space_integrals` only reduces arrays supplied by its caller. J-only HF
fitting combines fitted Coulomb with exact exchange; its energy is not generally
the expectation of the fully DF two-electron Hamiltonian. Compare identical
Hamiltonians and orbital bases before diagnosing variational errors.